In [ ]:
#| default_exp resumable

# Resumable upload (now part of `core`)

> Compatibility module. The resumable upload now lives in `dataset_archiver.core` and the `upload-dataset` command.

This module only re-exports it, so `python -m dataset_archiver.resumable ...` and `from dataset_archiver.resumable import ...`
keep working. It can be removed once nothing uses it. The test cells below use the functions from `core`.

In [ ]:
#| export
import sys
from dataset_archiver.core import *
from dataset_archiver.cli import build_cli, main

In [ ]:
#| export
#| eval: false
if __name__ == "__main__" and "ipykernel" not in sys.modules:
    sys.exit(main())

## Checks against the real server, from a notebook

The cells below were used to test the resumable upload on u-drive before the full ATLAS25 upload, and can be reused
whenever the code changes. They are marked `#| eval: false`, so `nbdev_test` skips them. Run them one at a time, from the
top, never with "Run All": step 3 writes to the server.

They use the three-file dataset in `data/`. `upload_dataset_to_webdav` and the other functions come from `core` through
the import above.

### 1. Credentials and session (asks for your password once)

Creates the login and one HTTP session reused by the following cells. To see straight away whether the password is right,
add `check_credentials(session, WEBDAV, auth)` at the end of the cell: it prints `🔑 Logged in` or explains the error.

In [ ]:
#| eval: false
from requests.auth import HTTPBasicAuth
from getpass import getpass

WEBDAV = "https://u-drive.unipi.it/remote.php/webdav"
USER, EMAIL = "a012197", "andrea.munafo@unipi.it"
auth = HTTPBasicAuth(USER, getpass(f"Password for {USER}: "))
session = make_session()
check_credentials(session, WEBDAV, auth)   # one PROPFIND on the WebDAV root; nothing is written

Password for a012197:  ········


🔑 Logged in to u-drive.unipi.it as a012197


### 2. Dry run on the small data folder (read-only)

You should see the 🔑 Logged in line, then "remote folder does not exist yet", a plan with 3 files to upload, and "(dry run) no MKCOL or PUT was sent" at the end

In [ ]:
#| eval: false
SUB = "/Users/andreamunafo/Downloads/CLAUDE/dataset-archiver/data"
md_test = dict(title="test", source="test", date_collected="n/a",
               description="resume test", responsible="Andrea Munafo")

res = upload_dataset_to_webdav(SUB, WEBDAV, "TEST_resume", md_test, USER, EMAIL,
                               auth=auth, session=session, dry_run=True)
res

📂 3 local files, 189.0 B (hidden files excluded)
🔑 Logged in to u-drive.unipi.it as a012197
🔢 Hashing folder contents with core.hash_folder (reads every byte; pass version_hash to skip)...
   hash = 1802d4f6 (0 s)

📁 Remote folder: https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/TEST_resume_20260927_1802d4f6
   To resume into this folder later: --archive-date 20260927 --version-hash 1802d4f6
🔎 Listing remote folder (PROPFIND Depth 1 per directory)...
   remote folder does not exist yet; it will be created

📋 Plan: 0 already on server (same size), 3 to upload (189.0 B), 0 on server with a DIFFERENT size (left alone; use --overwrite-mismatched to replace)
(dry run) would create folder: https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/TEST_resume_20260927_1802d4f6
(dry run) would create 1 subdirectories (one MKCOL each)
   + test_dataset/
(dry run) [1/3] would upload test_dataset/imu.csv (66.0 B)
(dry run) [2/3] would upload tes

{'url': 'https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/TEST_resume_20260927_1802d4f6',
 'folder': 'TEST_resume_20260927_1802d4f6',
 'archive_date': '20260927',
 'version_hash': '1802d4f6',
 'local': 3,
 'present': 0,
 'to_upload': 3,
 'mismatched': 0,
 'uploaded': 0,
 'failed': 0,
 'mkcol': 0,
 'aborted': False,
 'complete': False,
 'failure_log': None}

### 3. Real upload of data, then a resume, then verify (writes a small test folder)

Uses a scratch registry and log folder in `/tmp`, so the real `upload_registry.csv` is not touched. Set `DO_REAL_UPLOAD`
back to `False` after use, so the cell cannot upload again by accident.

If it gets to `OK:`, then the upload, the metadata files, the skip-on-resume and the size-checked verify all worked against
u-drive. Two further checks can be done by hand in the web interface:

- Resume after a partial upload: delete one of the three data files from the `TEST_resume_...` folder, then run the `r2`
  call again. It should report `uploaded=1`.
- Refusing to overwrite: replace one remote file with a different one of the same name, then run `r2` again. It should
  report a size mismatch, leave the file alone and skip the registry entry.

The tool never deletes, so remove the test folder by hand when you are done.

In [ ]:
#| eval: false
DO_REAL_UPLOAD = True
if DO_REAL_UPLOAD:
    kw = dict(auth=auth, session=session, registry_path="/tmp/test_registry.csv", log_dir="/tmp")
    r1 = upload_dataset_to_webdav(SUB, WEBDAV, "TEST_resume", dict(md_test), USER, EMAIL, **kw)
    assert r1["complete"], r1

    # Same folder, pinned: nothing should be sent the second time
    r2 = upload_dataset_to_webdav(SUB, WEBDAV, "TEST_resume", dict(md_test), USER, EMAIL,
                                  archive_date=r1["archive_date"], version_hash=r1["version_hash"], **kw)
    assert r2["uploaded"] == 0 and r2["present"] == r1["local"] and r2["mkcol"] == 0, r2

    v = verify_uploaded_dataset(SUB, r1["url"], auth, session=session, strict=True)
    print("OK:", r1["url"])

📂 3 local files, 189.0 B (hidden files excluded)
🔑 Logged in to u-drive.unipi.it as a012197
🔢 Hashing folder contents with core.hash_folder (reads every byte; pass version_hash to skip)...
   hash = 1802d4f6 (0 s)

📁 Remote folder: https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/TEST_resume_20260927_1802d4f6
   To resume into this folder later: --archive-date 20260927 --version-hash 1802d4f6
🔎 Listing remote folder (PROPFIND Depth 1 per directory)...
   remote has 6 files in 1 subdirectories

📋 Plan: 2 already on server (same size), 1 to upload (66.0 B), 0 on server with a DIFFERENT size (left alone; use --overwrite-mismatched to replace)
[1/1] ✅ test_dataset/imu.csv (66.0 B, 0.5 s)  141.1 B/s  ETA 0 min
⬆️ Sidecars: metadata.json, manifest.csv, LICENSE.txt, README.txt
   metadata.json: already on server, not overwritten
   manifest.csv: already on server, not overwritten
   LICENSE.txt: already on server, not overwritten
   README.txt: already on server,

### 4. Dry run of a resume (read-only)

Fill in the label, date and hash of the upload to resume (from the first lines of its output, or from the folder name on
u-drive, `LABEL_YYYYMMDD_HASH`) and the metadata used then. The plan should show the files already on the server and
none with a different size. With a wrong pin it stops with "Remote folder not found" and creates nothing.

For the test dataset in `data/` uploaded on 27 September 2026, the values are `--archive-date 20260927` and
`--version-hash 1802d4f6`.

In [ ]:
#| eval: false
FULL = "/Users/andreamunafo/Downloads/CLAUDE/dataset-archiver/data"
md_full = dict(title="test", source="test", date_collected="n/a",
               description="resume test", responsible="Andrea Munafo")

res = upload_dataset_to_webdav(FULL, WEBDAV, "TEST_resume", md_full, USER, EMAIL,
                               auth=auth, session=session,
                               archive_date="20260927", version_hash="1802d4f6", dry_run=True)
print(res["present"], res["to_upload"], res["mismatched"])   # expect about 4616, 31624, 0

📂 3 local files, 189.0 B (hidden files excluded)
🔑 Logged in to u-drive.unipi.it as a012197

📁 Remote folder: https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/TEST_resume_20260927_1802d4f6
   To resume into this folder later: --archive-date 20260927 --version-hash 1802d4f6
🔎 Listing remote folder (PROPFIND Depth 1 per directory)...
   remote has 7 files in 1 subdirectories

📋 Plan: 3 already on server (same size), 0 to upload (0.0 B), 0 on server with a DIFFERENT size (left alone; use --overwrite-mismatched to replace)
(dry run) would create 0 subdirectories (one MKCOL each)
(dry run) no MKCOL or PUT was sent
3 0 0


### 5. Verify the upload as it stands (read-only)

Compares the local folder with the remote one by name and size. For a partial upload, `missing` lists what is still to be
sent; `wrong_size` and `extra` should be empty.

In [ ]:
#| eval: false
v = verify_uploaded_dataset(FULL, res["url"], auth, session=session)
len(v["missing"]), v["wrong_size"], sorted(v["extra"])[:20]

🔑 Logged in to u-drive.unipi.it as a012197
🔍 Verifying 3 local files against https://u-drive.unipi.it/remote.php/webdav/Robotics-Research-Lab/archival_data/TEST_resume_20260927_1802d4f6
✅ Matched (name and size) : 3
❌ Missing on server       : 0
📏 Size differs            : 0
❔ Size not reported       : 0
⚠️  Only on server          : 0
📄 Sidecars missing        : none

📦 Verification complete.


(0, [], [])